In [1]:
%%writefile app.py
import streamlit as st
from utils.styling import inject_base_css, banner, note, footer, chip
from utils.db import init_db, seed_demo_borrowers

st.set_page_config(page_title="KhataSetu Finance", page_icon="🪙", layout="wide")
init_db()
seed_demo_borrowers(25)   # change to 1000 to load the full dataset
inject_base_css()

st.markdown(
    '<div style="color:#1F6B4A;font-weight:600;">SME LENDING &nbsp;|&nbsp; SEED ROUND &nbsp;|&nbsp; PROTOTYPE</div>',
    unsafe_allow_html=True,
)
st.title("KhataSetu Finance")
st.subheader("Turning the village ledger into a formal credit line.")
st.write(
    "A GST + UPI + Account Aggregator underwriting engine for New-to-Credit trader "
    "and micro-manufacturing MSMEs in rural and semi-urban India."
)

st.markdown("---")
banner("EXECUTIVE SUMMARY — Bridging India's Last-Mile Credit Gap With Data, Not Collateral")

c1, c2, c3, c4 = st.columns(4)
with c1:
    st.markdown('<div class="ks-card"><b>🧩 Problem</b><br>83% of India\'s 8.7 crore MSMEs '
                'have never accessed formal credit — deepest among rural/semi-urban traders '
                'with no bureau file.</div>', unsafe_allow_html=True)
with c2:
    st.markdown('<div class="ks-card"><b>💡 Solution</b><br>A revolving GST- and UPI-linked '
                'working capital line, underwritten on cash-flow data — no balance sheet required.</div>',
                unsafe_allow_html=True)
with c3:
    st.markdown('<div class="ks-card"><b>🏦 Model</b><br>Capital-light NBFC-LSP on a co-lending '
                'chassis — 10% own-book skin, 90% bank capital.</div>', unsafe_allow_html=True)
with c4:
    st.markdown('<div class="ks-card"><b>🎯 Ask</b><br>Rs 18 crore seed to reach Rs 40 crore AUM '
                'and 2,500 borrowers in Year 1.</div>', unsafe_allow_html=True)

st.markdown("---")
note("This is a working prototype of the KhataSetu product experience described in the "
     "investor pitch deck. Choose a persona below to explore the app.")

col1, col2 = st.columns(2)
with col1:
    st.markdown("### 👤 I'm a Borrower")
    st.write("Walk through the borrower journey: language selection → data consent → "
             "e-KYC → credit offer → e-sign & disbursement → repayment dashboard.")
    if st.button("Start Borrower Journey →", type="primary", use_container_width=True):
        st.switch_page("pages/1_Language_Onboarding.py")

with col2:
    st.markdown("### 🏦 I'm the Lender / Ops Team")
    st.write("View the portfolio: collections & NPA early-warning dashboard, and "
             "investor-facing business metrics (AUM build, unit economics).")
    b1, b2 = st.columns(2)
    with b1:
        if st.button("Ops Dashboard →", use_container_width=True):
            st.switch_page("pages/7_Lender_Ops_Dashboard.py")
    with b2:
        if st.button("Investor Metrics →", use_container_width=True):
            st.switch_page("pages/8_Investor_Metrics.py")

st.markdown("---")
chip("GST"); chip("UPI"); chip("Account Aggregator"); chip("e-Way Bill"); chip("Co-Lending"); chip("RBI Compliant")

footer()

Overwriting app.py


In [2]:
%%writefile pages/1_Language_Onboarding.py
import streamlit as st
from utils.styling import inject_base_css, banner, footer

st.set_page_config(page_title="KhataSetu | Onboarding", page_icon="🌐", layout="centered")
inject_base_css()

banner("TECH & PRODUCT — Onboarding & Language Selection")

st.markdown("### KhataSetu")
st.markdown("#### Apna Vyapar, Apni Bhasha")
st.caption("Choose your language / भाषा चुनें")

languages = ["Hindi / हिंदी", "English", "Gujarati / ગુજરાતી", "Marwari / मारवाड़ी"]
choice = st.radio(" ", languages, index=0, label_visibility="collapsed")

st.markdown(
    f'<div class="ks-whatsapp-btn">💬 Continue on WhatsApp ({choice.split(" / ")[0]})</div>',
    unsafe_allow_html=True,
)

st.write("")
with st.expander("Why this screen matters"):
    st.write(
        "- Vernacular-first design (Hindi + 3 regional languages) removes the biggest drop-off "
        "point for Tier 3-6 users.\n"
        "- WhatsApp Business API entry point meets borrowers where they already are, avoiding "
        "a separate app-install step at first contact.\n"
        "- A single, uncluttered choice up front — no forms, no jargon — builds trust before "
        "any data is asked for."
    )

st.markdown("---")
st.markdown("##### Tell us about your business")
col1, col2 = st.columns(2)
with col1:
    business_name = st.text_input("Business name", placeholder="e.g. Sharma Hardware Store")
    sector = st.selectbox("Sector", ["Trading", "Micro-Manufacturing"])
with col2:
    gstin = st.text_input("GSTIN", placeholder="07ABCDE1234F1Z5", max_chars=15)
    cluster = st.selectbox("Cluster / Location", ["UP - Moradabad", "Rajasthan - Bhilwara", "Gujarat - Morbi"])

turnover_band = st.select_slider(
    "Approximate annual turnover",
    options=["< Rs 40L", "Rs 40L-1Cr", "Rs 1-3Cr", "Rs 3-5Cr", "> Rs 5Cr"],
    value="Rs 1-3Cr",
)

if st.button("Continue →", type="primary"):
    if not business_name or not gstin:
        st.error("Please enter your business name and GSTIN to continue.")
    else:
        st.session_state["onboarding"] = {
            "language": choice.split(" / ")[0],
            "business_name": business_name,
            "gstin": gstin,
            "sector": sector,
            "cluster": cluster,
            "turnover_band": turnover_band,
        }
        st.switch_page("pages/2_Data_Consent.py")

footer()

Writing pages/1_Language_Onboarding.py


In [3]:
%%writefile pages/2_Data_Consent.py
import streamlit as st
from utils.styling import inject_base_css, banner, footer

st.set_page_config(page_title="KhataSetu | Consent", page_icon="🔐", layout="centered")
inject_base_css()

if "onboarding" not in st.session_state:
    st.warning("Please start from the onboarding step.")
    if st.button("← Back to Onboarding"):
        st.switch_page("pages/1_Language_Onboarding.py")
    st.stop()

banner("TECH & PRODUCT — GST, UPI & Account Aggregator Consent")
st.markdown("### Data Consent")
st.caption(f"Business: {st.session_state['onboarding']['business_name']}")

c1, c2 = st.columns(2)
with c1:
    gst_consent = st.checkbox("📄 GST Returns — Last 12 months", value=True)
    aa_consent = st.checkbox("🏦 Bank Account (AA) — via Finvu / OneMoney", value=True)
with c2:
    upi_consent = st.checkbox("💳 UPI Settlement History — 6 months", value=True)
    eway_consent = st.checkbox("🚚 E-Way Bill Trail — Dispatch data", value=True)

st.write("")
all_granted = gst_consent and aa_consent and upi_consent and eway_consent
if st.button("🔒 Grant Secure Consent", type="primary", disabled=not all_granted):
    st.session_state["consent"] = {
        "gst": gst_consent, "aa": aa_consent, "upi": upi_consent, "eway": eway_consent,
    }
    st.success("Consent granted. Pulling your data securely...")
    st.switch_page("pages/3_eKYC_Verification.py")

if not all_granted:
    st.info("All four consents are needed to build your cash-flow score.")

with st.expander("Why this screen matters"):
    st.write(
        "- Purpose-limited, granular consent per RBI's Account Aggregator and Digital Lending "
        "Directions — the borrower sees exactly what is pulled and why.\n"
        "- GSTN's status as a live AA Financial Information Provider lets us pull GST data "
        "through the same consent flow as bank data — one screen, one OTP.\n"
        "- This is the single most important screen in the funnel: it converts a manual "
        "PDF-upload nightmare into a sub-5-minute, revocable, bank-grade consent."
    )

footer()

Writing pages/2_Data_Consent.py


In [4]:
%%writefile pages/3_eKYC_Verification.py
import streamlit as st
from utils.styling import inject_base_css, banner, footer

st.set_page_config(page_title="KhataSetu | e-KYC", page_icon="🪪", layout="centered")
inject_base_css()

if "consent" not in st.session_state:
    st.warning("Please complete the consent step first.")
    if st.button("← Back to Consent"):
        st.switch_page("pages/2_Data_Consent.py")
    st.stop()

banner("TECH & PRODUCT — Document Upload & e-KYC")
st.markdown("### Verify Identity")
st.progress(2 / 4, text="Step 2 of 4")

aadhaar = st.text_input("Aadhaar Number", placeholder="XXXX XXXX 4821", max_chars=14)
pan = st.text_input("PAN (auto-fetched)", value="ABCDE1234F", disabled=True)
st.caption("✅ PAN verified")

st.write("")
st.markdown("**Live selfie**")
selfie = st.camera_input("Tap to capture live selfie")

st.write("")
if st.button("✅ Verify & Continue", type="primary"):
    if not aadhaar or len(aadhaar.replace(" ", "")) < 4:
        st.error("Please enter a valid Aadhaar number.")
    else:
        st.session_state["ekyc"] = {
            "aadhaar_masked": "XXXX XXXX " + aadhaar.replace(" ", "")[-4:],
            "pan": pan,
            "selfie_captured": selfie is not None,
        }
        st.success("Identity verified.")
        st.switch_page("pages/4_Credit_Offer.py")

with st.expander("Why this screen matters"):
    st.write(
        "- Aadhaar e-KYC + PAN auto-fetch removes physical document uploads entirely — critical "
        "where scanning/uploading PDFs is the single biggest funnel drop-off.\n"
        "- Live selfie liveness-check satisfies RBI KYC norms while remaining a one-tap action "
        "on a basic Android smartphone.\n"
        "- A visible 4-step progress bar sets expectations for a borrower unfamiliar with "
        "digital loan applications."
    )

footer()

Writing pages/3_eKYC_Verification.py


In [5]:
%%writefile pages/4_Credit_Offer.py
import streamlit as st
import pandas as pd
from utils.styling import inject_base_css, banner, footer
from utils.synthetic_data import generate_borrower_data
from utils.scoring_engine import score_borrower
from utils.db import insert_borrower
import json
from datetime import datetime

st.set_page_config(page_title="KhataSetu | Credit Offer", page_icon="💰", layout="centered")
inject_base_css()

if "ekyc" not in st.session_state:
    st.warning("Please complete e-KYC first.")
    if st.button("← Back to e-KYC"):
        st.switch_page("pages/3_eKYC_Verification.py")
    st.stop()

banner("TECH & PRODUCT — Alt-Data Underwriting Engine & Credit Offer")
st.markdown("### Your Offer")
st.progress(3 / 4, text="Step 3 of 4")

if "synthetic_data" not in st.session_state:
    with st.spinner("Pulling GST, UPI, e-way bill & AA data... provisional offer in under 15 minutes"):
        st.session_state["synthetic_data"] = generate_borrower_data()

synth = st.session_state["synthetic_data"]
result = score_borrower(synth)
st.session_state["offer_result"] = result

st.markdown("#### Cash-Flow Scorecard")
comp = result["components"]
df = pd.DataFrame({
    "Signal": ["GST Filing Consistency", "UPI Inflow Trend", "E-Way Bill Momentum",
               "AA Bank Cash-Flow", "Bureau (if any)"],
    "Score (0-100)": [comp["gst"], comp["upi"], comp["eway"], comp["aa"], comp["bureau"]],
})
st.bar_chart(df.set_index("Signal"))
st.caption(f"Composite Cash-Flow Score: **{result['score']}/100** — {result['risk_band']}")

st.markdown("---")

if result["sanctioned_limit"] <= 0:
    st.error("Based on current cash-flow signals, this application needs manual credit review. "
              "No automated offer can be extended at this time.")
    footer()
    st.stop()

c1, c2, c3 = st.columns(3)
c1.metric("Sanctioned Limit", f"₹{result['sanctioned_limit']:,.0f}")
c2.metric("Interest rate (blended)", f"{result['interest_rate']}% p.a.")
c3.metric("APR (all-in)", f"{result['apr']}%")

c4, c5 = st.columns(2)
c4.metric("Processing fee", f"{result['processing_fee_pct']}%")
c5.metric("Tenure", f"{result['tenure_months']} months, revolving")

st.markdown("#### Key Facts Statement (KFS)")
kfs_df = pd.DataFrame({
    "Field": ["Sanctioned Limit", "Interest Rate (blended)", "Processing Fee", "Tenure", "APR (all-in)",
              "Security", "Lenders behind this loan"],
    "Value": [f"₹{result['sanctioned_limit']:,.0f}", f"{result['interest_rate']}% p.a.",
              f"{result['processing_fee_pct']}% of sanctioned limit", "12 months, revolving",
              f"{result['apr']}%", "Hypothecation of stock/receivables + UPI Autopay mandate",
              "Bank partner (90%) + KhataSetu NBFC-LSP (10%) — single blended rate shown"],
})
st.table(kfs_df.set_index("Field"))
st.caption("Shown in full, in plain language, as mandated by RBI's Digital Lending Directions, 2025. "
           "A cooling-off window applies per KhataSetu's board-approved policy.")

if st.button("✅ Accept Offer", type="primary"):
    borrower_data = {
        "business_name": st.session_state["onboarding"]["business_name"],
        "gstin": st.session_state["onboarding"]["gstin"],
        "sector": st.session_state["onboarding"]["sector"],
        "turnover_band": st.session_state["onboarding"]["turnover_band"],
        "cluster": st.session_state["onboarding"]["cluster"],
        "language": st.session_state["onboarding"]["language"],
        "aadhaar_masked": st.session_state["ekyc"]["aadhaar_masked"],
        "pan": st.session_state["ekyc"]["pan"],
        "kyc_verified": 1,
        "consent_json": json.dumps(st.session_state["consent"]),
        "synthetic_data_json": json.dumps(synth),
        "score": result["score"],
        "risk_band": result["risk_band"],
        "sanctioned_limit": result["sanctioned_limit"],
        "available_limit": result["sanctioned_limit"],
        "interest_rate": result["interest_rate"],
        "processing_fee_pct": result["processing_fee_pct"],
        "apr": result["apr"],
        "offer_accepted": 1,
        "disbursed": 0,
        "disbursed_amount": 0,
        "next_emi": 0,
        "next_emi_due": "",
        "autopay": 1,
        "npa_flag": 0,
        "dpd": 0,
        "created_at": datetime.now().isoformat(),
    }
    borrower_id = insert_borrower(borrower_data)
    st.session_state["borrower_id"] = borrower_id
    st.success("Offer accepted!")
    st.switch_page("pages/5_eSign_Disbursement.py")

with st.expander("Why this screen matters"):
    st.write(
        "- The Key Facts Statement is shown in full, in plain language, exactly as mandated by "
        "RBI's Digital Lending Directions — APR, fees and tenure upfront, no fine print.\n"
        "- One blended rate is shown even though two lenders (bank + KhataSetu) sit behind the "
        "loan, as required under the Co-Lending Directions, 2025.\n"
        "- A visible cooling-off window lets the borrower exit penalty-free within the "
        "RE's board-approved period."
    )

footer()

Writing pages/4_Credit_Offer.py


In [6]:
%%writefile pages/5_eSign_Disbursement.py
import streamlit as st
from datetime import datetime, timedelta
from utils.styling import inject_base_css, banner, footer
from utils.db import update_borrower, add_repayment

st.set_page_config(page_title="KhataSetu | e-Sign", page_icon="✍️", layout="centered")
inject_base_css()

if "borrower_id" not in st.session_state:
    st.warning("Please accept your offer first.")
    if st.button("← Back to Offer"):
        st.switch_page("pages/4_Credit_Offer.py")
    st.stop()

banner("TECH & PRODUCT — e-Sign & Disbursement")
st.markdown("### Almost Done")
st.progress(4 / 4, text="Step 4 of 4")

result = st.session_state["offer_result"]
sanctioned = result["sanctioned_limit"]

st.markdown("#### Loan Agreement")
st.write("Aadhaar e-Sign | OTP verification")
otp = st.text_input("Enter OTP sent to your registered mobile", max_chars=6, placeholder="123456")

if "esigned" not in st.session_state:
    st.session_state["esigned"] = False

if not st.session_state["esigned"]:
    if st.button("Verify OTP & e-Sign", type="primary"):
        if otp and len(otp) >= 4:
            st.session_state["esigned"] = True
            st.rerun()
        else:
            st.error("Please enter a valid OTP.")
else:
    st.success("✅ e-signature captured")
    draw_amount = st.slider(
        "How much would you like to draw now from your sanctioned limit?",
        min_value=int(sanctioned * 0.2), max_value=sanctioned, value=int(sanctioned * 0.5), step=5000,
    )
    if st.button("💸 Disburse Funds", type="primary"):
        next_emi = round(draw_amount / 9 * (1 + result["interest_rate"] / 100 / 12 * 6), -2)
        next_due = (datetime.now() + timedelta(days=30)).strftime("%d %b %Y")
        update_borrower(st.session_state["borrower_id"], {
            "offer_accepted": 1,
            "disbursed": 1,
            "disbursed_amount": draw_amount,
            "available_limit": sanctioned - draw_amount,
            "next_emi": next_emi,
            "next_emi_due": next_due,
        })
        add_repayment(st.session_state["borrower_id"], 1, next_emi, "Upcoming", next_due)
        st.session_state["disbursed_amount"] = draw_amount
        st.session_state["masked_account"] = "XX" + st.session_state["ekyc"]["aadhaar_masked"][-4:]
        st.balloons()
        st.markdown(
            f'<div class="ks-card">✅ <b>Funds Credited</b><br>'
            f'₹{draw_amount:,.0f} credited to A/c {st.session_state["masked_account"]} via escrow</div>',
            unsafe_allow_html=True,
        )
        if st.button("View Repayment Schedule →", type="primary"):
            st.switch_page("pages/6_My_Vyapar_Line.py")

with st.expander("Why this screen matters"):
    st.write(
        "- Aadhaar e-Sign closes the loop without a single physical signature or branch visit — "
        "core to the sub-48-hour promise.\n"
        "- Disbursal moves through an escrow account straight to the borrower's own bank account, "
        "never to a third party, per RBI's digital lending disbursal rule.\n"
        "- A clear confirmation moment builds trust for a first-time formal borrower and sets up "
        "the repayment relationship that follows."
    )

footer()

Writing pages/5_eSign_Disbursement.py


In [7]:
%%writefile pages/6_My_Vyapar_Line.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from utils.styling import inject_base_css, banner, footer
from utils.db import get_borrower, get_repayments, add_repayment, update_borrower, get_all_borrowers

st.set_page_config(page_title="KhataSetu | My Vyapar Line", page_icon="📊", layout="centered")
inject_base_css()

banner("MY VYAPAR LINE — Repayment Dashboard")

if "borrower_id" not in st.session_state:
    all_b = get_all_borrowers()
    if not all_b:
        st.info("No borrowers yet. Complete the borrower journey first, or come back after seeding demo data.")
        st.stop()
    options = {f'{b["business_name"]} (ID {b["id"]})': b["id"] for b in all_b if b["disbursed"]}
    if not options:
        st.info("No disbursed borrowers yet.")
        st.stop()
    picked = st.selectbox("View dashboard for:", list(options.keys()))
    st.session_state["borrower_id"] = options[picked]

borrower = get_borrower(st.session_state["borrower_id"])
if not borrower or not borrower["disbursed"]:
    st.warning("This borrower hasn't been disbursed yet.")
    st.stop()

st.markdown(f"### {borrower['business_name']}")
st.caption(f"{borrower['cluster']} · {borrower['sector']} · Risk band: {borrower['risk_band']}")

available = borrower["available_limit"]
sanctioned = borrower["sanctioned_limit"]
st.progress(min(available / sanctioned, 1.0), text=f"Available Limit: ₹{available:,.0f} of ₹{sanctioned:,.0f}")

c1, c2, c3 = st.columns(3)
c1.metric("Next EMI", f"₹{borrower['next_emi']:,.0f}")
c2.metric("Due", borrower["next_emi_due"] or "—")
c3.metric("Autopay", "UPI Autopay ON" if borrower["autopay"] else "OFF")

st.markdown("---")
st.markdown("#### Repayment History")
reps = get_repayments(borrower["id"])
if reps:
    df = pd.DataFrame(reps)[["cycle_no", "amount", "status", "paid_on"]]
    df.columns = ["Cycle", "Amount (₹)", "Status", "Date"]
    st.bar_chart(df.set_index("Cycle")["Amount (₹)"])
    st.dataframe(df, use_container_width=True, hide_index=True)
else:
    st.info("No repayment cycles yet.")

st.caption(
    "A visible repayment-history bar chart reinforces the credit-ladder story: on-time cycles "
    "unlock a higher limit at a lower rate. This dashboard doubles as the early-warning surface — "
    "a missed bar here is the same signal that triggers the collections workflow."
)

st.markdown("---")
col1, col2 = st.columns(2)
with col1:
    if st.button("💳 Repay Now", type="primary", use_container_width=True):
        add_repayment(borrower["id"], len(reps) + 1, borrower["next_emi"], "Paid on time",
                      datetime.now().strftime("%d %b %Y"))
        update_borrower(borrower["id"], {
            "available_limit": min(sanctioned, available + borrower["next_emi"] * 0.6),
            "next_emi_due": (datetime.now() + timedelta(days=30)).strftime("%d %b %Y"),
            "dpd": 0,
        })
        st.success("Payment recorded. Your available limit has been updated.")
        st.rerun()
with col2:
    if st.button("⬆️ Raise Limit", use_container_width=True):
        on_time = sum(1 for r in reps if r["status"] == "Paid on time")
        if reps and on_time / len(reps) >= 0.7:
            new_limit = round(sanctioned * 1.15, -3)
            update_borrower(borrower["id"], {
                "sanctioned_limit": new_limit,
                "available_limit": available + (new_limit - sanctioned),
                "interest_rate": max(16.0, borrower["interest_rate"] - 0.5),
            })
            st.success(f"Great repayment history! Limit raised to ₹{new_limit:,.0f} and rate improved — "
                       "the credit ladder in action.")
            st.rerun()
        else:
            st.warning("Limit increases unlock after a consistent on-time repayment history.")

footer()

Writing pages/6_My_Vyapar_Line.py


In [8]:
%%writefile pages/7_Lender_Ops_Dashboard.py
import streamlit as st
import pandas as pd
from utils.styling import inject_base_css, banner, subbanner, footer
from utils.db import get_all_borrowers, get_alerts

st.set_page_config(page_title="KhataSetu | Ops Dashboard", page_icon="🏦", layout="wide")
inject_base_css()

banner("LENDER / OPS DASHBOARD — Portfolio, Collections & NPA Management")

borrowers = get_all_borrowers()
if not borrowers:
    st.info("No borrowers yet. Visit the Home page to seed demo data.")
    st.stop()

df = pd.DataFrame(borrowers)
disbursed_df = df[df["disbursed"] == 1]

total_aum = disbursed_df["disbursed_amount"].sum()
active_borrowers = len(disbursed_df)
npa_count = disbursed_df["npa_flag"].sum()
npa_rate = (npa_count / active_borrowers * 100) if active_borrowers else 0

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total AUM", f"₹{total_aum:,.0f}")
c2.metric("Active Borrowers", f"{active_borrowers:,}")
c3.metric("NPA Rate", f"{npa_rate:.1f}%", delta=f"{npa_rate - 4:.1f} pts vs 4% target",
          delta_color="inverse")
c4.metric("Avg Sanctioned Limit", f"₹{df['sanctioned_limit'].mean():,.0f}")

st.markdown("---")

col_a, col_b = st.columns([1.3, 1])
with col_a:
    subbanner("Cluster-wise Portfolio")
    cluster_summary = disbursed_df.groupby("cluster").agg(
        borrowers=("id", "count"), aum=("disbursed_amount", "sum")
    ).reset_index()
    st.bar_chart(cluster_summary.set_index("cluster")["aum"])
    st.dataframe(cluster_summary.rename(columns={"cluster": "Cluster", "borrowers": "Borrowers", "aum": "AUM (₹)"}),
                 hide_index=True, use_container_width=True)

with col_b:
    subbanner("Co-Lending Split (90 / 10)")
    bank_share = total_aum * 0.9
    khatasetu_share = total_aum * 0.1
    split_df = pd.DataFrame({"Partner": ["Bank / SFB (90%)", "KhataSetu NBFC-LSP (10%)"],
                              "Amount": [bank_share, khatasetu_share]})
    st.bar_chart(split_df.set_index("Partner"))
    st.caption(f"Bank @ ~9.5% cost of funds: ₹{bank_share:,.0f}  ·  "
               f"KhataSetu @ ~13%: ₹{khatasetu_share:,.0f}")

st.markdown("---")
subbanner("Risk Band Distribution")
risk_summary = df["risk_band"].value_counts().reset_index()
risk_summary.columns = ["Risk Band", "Count"]
st.bar_chart(risk_summary.set_index("Risk Band"))

st.markdown("---")
banner("Early Warning Signals — Leading Indicators")
st.write(
    "🔻 UPI inflow decline >20% (30d)  ·  📅 GST filing delayed  ·  🚚 E-way bill volume drop  ·  "
    "🔁 First UPI Autopay bounce"
)

alerts = get_alerts()
if alerts:
    alerts_df = pd.DataFrame(alerts)
    borrower_lookup = {b["id"]: b["business_name"] for b in borrowers}
    alerts_df["business_name"] = alerts_df["borrower_id"].map(borrower_lookup)
    alerts_df = alerts_df[["business_name", "alert_type", "detail", "stage", "created_at"]]
    alerts_df.columns = ["Borrower", "Alert Type", "Detail", "Escalation Stage", "Flagged At"]
    st.dataframe(alerts_df, hide_index=True, use_container_width=True)
else:
    st.success("No active early-warning alerts.")

st.markdown("---")
subbanner("Escalation Ladder")
ladder = pd.DataFrame({
    "Stage": ["Day 0-2", "Day 3-7", "Day 8-15", "Day 16-30", "30+ DPD"],
    "Action": ["Automated SMS/WhatsApp nudge", "Cluster manager call", "On-ground field visit",
               "Restructuring / OTS offer", "Bureau reporting; joint asset classification"],
    "Borrowers at this stage": [
        len(disbursed_df[(disbursed_df["dpd"] > 0) & (disbursed_df["dpd"] <= 2)]),
        len(disbursed_df[(disbursed_df["dpd"] > 2) & (disbursed_df["dpd"] <= 7)]),
        len(disbursed_df[(disbursed_df["dpd"] > 7) & (disbursed_df["dpd"] <= 15)]),
        len(disbursed_df[(disbursed_df["dpd"] > 15) & (disbursed_df["dpd"] <= 30)]),
        len(disbursed_df[disbursed_df["dpd"] > 30]),
    ],
})
st.table(ladder.set_index("Stage"))

st.markdown("---")
subbanner("Borrower Register")
show_cols = ["id", "business_name", "cluster", "sector", "risk_band", "sanctioned_limit",
             "disbursed_amount", "dpd", "npa_flag"]
reg = df[show_cols].rename(columns={
    "id": "ID", "business_name": "Business", "cluster": "Cluster", "sector": "Sector",
    "risk_band": "Risk Band", "sanctioned_limit": "Sanctioned (₹)",
    "disbursed_amount": "Disbursed (₹)", "dpd": "DPD", "npa_flag": "NPA",
})
st.dataframe(reg, hide_index=True, use_container_width=True)

footer()

Writing pages/7_Lender_Ops_Dashboard.py


In [9]:
%%writefile pages/8_Investor_Metrics.py
import streamlit as st
import pandas as pd
from utils.styling import inject_base_css, banner, subbanner, footer
from utils.db import get_all_borrowers

st.set_page_config(page_title="KhataSetu | Investor Metrics", page_icon="📈", layout="wide")
inject_base_css()

banner("BUSINESS PLAN — 3-Year AUM Build & Unit Economics")

borrowers = get_all_borrowers()
df = pd.DataFrame(borrowers) if borrowers else pd.DataFrame()

st.markdown("#### 3-Year AUM Build & Path to EBITDA Positive")
plan_df = pd.DataFrame({
    "Year": ["Year 1", "Year 2", "Year 3"],
    "Borrowers": [2500, 9000, 22000],
    "AUM (₹ Cr)": [40, 180, 550],
    "Clusters": [3, 8, 15],
})
c1, c2 = st.columns([1.4, 1])
with c1:
    st.bar_chart(plan_df.set_index("Year")["AUM (₹ Cr)"])
with c2:
    st.table(plan_df.set_index("Year"))
st.caption("EBITDA turns positive around Month 30-32 (Q2 of Year 3) as NIM and fee income "
           "outpace cluster opex.")

st.markdown("---")
subbanner("Unit Economics")
u1, u2, u3, u4 = st.columns(4)
u1.metric("CAC", "₹3,200 - ₹4,000", help="Phygital: cluster manager + digital sourcing")
u2.metric("3-Yr LTV", "~₹28,000", help="Multi-cycle revolving relationship")
u3.metric("LTV : CAC", "~7-8x", help="After 2 renewal cycles")
u4.metric("Gross NPA Target", "< 4.0%", help="vs 5-7% industry typical")

st.markdown("---")
subbanner("Revenue Model — Four Streams, Anchored on Spread")
rev_df = pd.DataFrame({
    "Stream": ["Net Interest Margin", "Processing Fee", "Co-Lending Servicing Spread", "Cross-Sell (Yr 2+)"],
    "Detail": [
        "~5.5-6.0% on 10% own-book share (~17% blended rate less ~11% cost of funds)",
        "1.0-1.5% of sanctioned limit, on origination and renewal",
        "Sourcing/underwriting/collections fee on bank partner's 90% share",
        "Embedded GST-filing SaaS, invoice financing, credit-linked insurance",
    ],
})
st.table(rev_df.set_index("Stream"))

st.markdown("---")
subbanner("Live Prototype Snapshot (from seeded/demo data)")
if not df.empty:
    disbursed_df = df[df["disbursed"] == 1]
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Prototype AUM", f"₹{disbursed_df['disbursed_amount'].sum():,.0f}")
    m2.metric("Prototype Borrowers", f"{len(disbursed_df)}")
    m3.metric("Avg Score", f"{df['score'].mean():.1f}/100")
    m4.metric("Avg Interest Rate", f"{df['interest_rate'].mean():.1f}%")
else:
    st.info("No demo data yet.")

footer()

Writing pages/8_Investor_Metrics.py
